In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.datasets import load_wine

In [4]:
data = load_wine(as_frame=True)
df = data.frame

In [5]:
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [6]:
y = df['target']
X= df.drop(['target'], axis=1)

In [7]:
X.shape

(178, 13)

In [8]:
np.unique(y, return_counts=True)

(array([0, 1, 2]), array([59, 71, 48], dtype=int64))

In [9]:
X.isna().sum()

alcohol                         0
malic_acid                      0
ash                             0
alcalinity_of_ash               0
magnesium                       0
total_phenols                   0
flavanoids                      0
nonflavanoid_phenols            0
proanthocyanins                 0
color_intensity                 0
hue                             0
od280/od315_of_diluted_wines    0
proline                         0
dtype: int64

In [10]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.33, random_state=47, stratify = y, shuffle=True)

In [11]:
log_reg = LogisticRegression(n_jobs=-1)
log_reg.fit(X_train, y_train)
y_pred_log = log_reg.predict(X_test)

print(classification_report(y_test, y_pred_log))

              precision    recall  f1-score   support

           0       0.95      1.00      0.98        20
           1       1.00      0.91      0.95        23
           2       0.94      1.00      0.97        16

    accuracy                           0.97        59
   macro avg       0.96      0.97      0.97        59
weighted avg       0.97      0.97      0.97        59



In [12]:
rf_clf= RandomForestClassifier(n_estimators = 40, max_depth = 3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97        20
           1       0.95      0.91      0.93        23
           2       0.89      1.00      0.94        16

    accuracy                           0.95        59
   macro avg       0.95      0.95      0.95        59
weighted avg       0.95      0.95      0.95        59



In [13]:
dtree = DecisionTreeClassifier(max_depth=5)
dtree.fit(X_train, y_train)
y_pred_tree = dtree.predict(X_test)
print(classification_report(y_test, y_pred_tree))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97        20
           1       0.95      0.91      0.93        23
           2       0.89      1.00      0.94        16

    accuracy                           0.95        59
   macro avg       0.95      0.95      0.95        59
weighted avg       0.95      0.95      0.95        59



In [14]:
svm_clf = SVC()
svm_clf.fit(X_train, y_train)
y_pred_svm = svm_clf.predict(X_test)
print(classification_report(y_test, y_pred_svm))


              precision    recall  f1-score   support

           0       0.83      0.95      0.88        20
           1       0.61      0.96      0.75        23
           2       0.00      0.00      0.00        16

    accuracy                           0.69        59
   macro avg       0.48      0.64      0.54        59
weighted avg       0.52      0.69      0.59        59



In [15]:
from imblearn.over_sampling import SMOTE

In [16]:
smt = SMOTE()

In [17]:
import mlflow
import mlflow.sklearn

In [18]:
mlflow.set_tracking_uri('http://127.0.0.1:5000/')

In [19]:
mlflow.set_experiment('Haram thing experiment')

<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1779105070661, experiment_id='2', last_update_time=1779105070661, lifecycle_stage='active', name='Haram thing experiment', tags={}, trace_location=None, workspace='default'>

In [20]:
models = [
    ('Logistic Regression', LogisticRegression(n_jobs=-1), (X_train, y_train), (X_test, y_test)),
    ('Random Forest', RandomForestClassifier(n_estimators=20, max_depth=3), (X_train, y_train), (X_test, y_test)),
    ('Decision Tree', DecisionTreeClassifier(max_depth=3), (X_train, y_train), (X_test, y_test)),
    ('Support Vector Machine', SVC(), (X_train, y_train), (X_test, y_test))
]

In [21]:
reports = []

for model_name, each_model, train_set, test_set in models:
    each_model.fit(train_set[0], train_set[1])
    pred = each_model.predict(test_set[0])
    reports.append(classification_report(test_set[1], pred, output_dict=True))

In [23]:
for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model', model_name)
        mlflow.log_metric('Accuracy', report['accuracy'])
        mlflow.log_metric('Recall_class_0', report['0']['recall'])
        mlflow.log_metric('Recall_class_1', report['1']['recall'])
        mlflow.log_metric('Recall_class_2', report['2']['recall'])
        mlflow.sklearn.log_model(model, name='model')

2026/05/18 16:00:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/2/runs/596f6fc0c0a44dab9b663a62aa5f1846
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/05/18 16:01:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/2/runs/1daeecde65fb4d78add08c8154ff6329
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/05/18 16:01:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Decision Tree at: http://127.0.0.1:5000/#/experiments/2/runs/b223158c24d344ec951492777f642f5f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


2026/05/18 16:01:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Support Vector Machine at: http://127.0.0.1:5000/#/experiments/2/runs/8625c798c934410385f66125f66d0bb9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [24]:
mlflow.end_run()